In [1]:
"""
将单倍型频率和表型信息合并，并且做成可以直接用于机器学习的训练集和测试集
CatBoost可以读入缺失值的，其他的深度学习不可以，所以要另外构建数据集
"""
import os
import numpy as np
import pandas as pd

from sklearn import preprocessing

WORK_PATH = "/share/home/zhoujb/project/riceDataComb/processData/dataFromCowpea/"

In [22]:
df_matrix = pd.read_table(os.path.join(WORK_PATH, "phen_matrix_2.txt"))
df_per = pd.read_table(os.path.join(WORK_PATH, "cowpea_hap_per.txt"))
inte_sam_list = list(set(df_matrix["Accessions"].unique()).intersection(df_per["sample"].unique()))

df_matrix = df_matrix[df_matrix["Accessions"].isin(inte_sam_list)].copy()
df_per = df_per[df_per["sample"].isin(inte_sam_list)].copy()


df_merge = df_matrix.merge(df_per, left_on="Accessions", right_on="sample", how="left")
df_merge = df_merge.drop(columns=["sample"])

In [26]:
df_merge

,Accessions,region,Pod length,Pod Sugar content,Pod Starch content,Pod Protein content,Seed Sugar,Seed Starch,Seed Protein,TSW,...,Chr05_47844905_47894811,Chr06_37401985_37467327,Chr06_41055041_41187272,Chr11_2731866_3024746,Chr10_52677456_52690014,Chr11_41910285_41974381,Chr04_2872265_2930398,Chr03_14880958_14944681,Chr03_56687574_56868067,Chr03_56866253_56866253
0,D365,HZ,34.15000,13.227273,13.077019,22.594056,93.127821,332.589113,259.378125,131.00245,...,0.011628,0.002907,0.002907,0.002907,0.002907,0.002907,0.322674,0.305233,0.005814,0.875000
1,D366,HZ,49.74375,11.517036,9.184116,23.338749,100.376071,383.131156,277.807292,133.33695,...,0.072674,0.002907,0.002907,0.002907,0.116279,0.002907,0.005814,0.058140,0.002907,0.875000
2,D367,HZ,67.97500,14.012441,12.248932,24.375913,81.751904,291.036363,285.440625,96.50000,...,0.258721,0.005814,0.002907,0.005814,0.116279,0.002907,0.002907,0.305233,0.002907,0.875000
3,D369,HZ,50.22500,12.304999,10.877537,24.108691,77.230174,317.698957,268.146875,112.35710,...,0.258721,0.008721,0.002907,0.046512,0.162791,0.052326,0.002907,0.043605,0.252907,0.875000
4,D370,HZ,52.37500,13.314397,14.040737,20.998941,82.590102,370.513334,259.130208,117.64265,...,0.072674,0.055233,0.005814,0.345930,0.116279,0.098837,0.002907,0.014535,0.252907,0.875000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
683,D724,GZ,14.80000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,0.258721,0.002907,0.002907,0.002907,0.116279,0.002907,0.002907,0.002907,0.002907,0.008721
684,D725,GZ,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,0.002907,0.002907,0.002907,0.002907,0.002907,0.002907,0.002907,0.002907,0.002907,0.037791
685,D726,GZ,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,0.002907,0.002907,0.002907,0.002907,0.002907,0.002907,0.002907,0.002907,0.002907,0.037791
686,D727,GZ,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,0.002907,0.308140,0.002907,0.002907,0.002907,0.002907,0.322674,0.023256,0.002907,0.875000


In [27]:
for item in ['Pod length', 'Pod Sugar content', 'Pod Starch content', 'Pod Protein content', 
             'Seed Sugar', 'Seed Starch', 'Seed Protein', 'TSW']:
    tmp_data = df_merge.copy().dropna(subset=item)
    # 保留 NaN比例 <=50% 的列
    tmp_data = tmp_data.loc[:, tmp_data.isna().mean() <= 0.5].copy()
    
    tmp_data = tmp_data.rename(columns={x:"fea_{}".format(x) for x in tmp_data.columns if x not in [item, 'Accessions', 'region']})
    tmp_data.to_csv(os.path.join(WORK_PATH, "trainTestData/data-{}.txt".format(item)), sep="\t", index=0)

    # 每个 location + year 抽样 10%
    test_df = (
        tmp_data.groupby(["region"], group_keys=False)
        .apply(lambda x: x.sample(frac=0.1, random_state=42))
    )
    # 剩余为训练集
    train_df = tmp_data.drop(test_df.index).copy()

    feat_col = [c for c in tmp_data.columns if c.startswith("fea_")]
    target_col = item

    scale_tool = preprocessing.StandardScaler()
    scale_tool.fit(train_df.loc[:, feat_col])
    train_df.loc[:, feat_col] = scale_tool.transform(train_df.loc[:, feat_col])
    test_df.loc[:, feat_col] = scale_tool.transform(test_df.loc[:, feat_col])

    train_df.to_csv(os.path.join(WORK_PATH, "trainTestData/train-{}.txt".format(item)), sep="\t", index=0)
    test_df.to_csv(os.path.join(WORK_PATH, "trainTestData/test-{}.txt".format(item)), sep="\t", index=0)
    
print("ALL DONE")

/tmp/ipykernel_276644/215545458.py:13: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda x: x.sample(frac=0.1, random_state=42))
/tmp/ipykernel_276644/215545458.py:13: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda x: x.sample(frac=0.1, random_state=42))
/tmp/ipykernel_276644/215545458.py:13: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is

ALL DONE


/tmp/ipykernel_276644/215545458.py:13: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda x: x.sample(frac=0.1, random_state=42))


In [28]:
for item in ['Pod length', 'Pod Sugar content', 'Pod Starch content', 'Pod Protein content', 
             'Seed Sugar', 'Seed Starch', 'Seed Protein', 'TSW']:

    tmp_data = df_merge.copy().dropna(subset=[item])

    # 保留 NaN比例 <=50% 的列
    tmp_data = tmp_data.loc[:, tmp_data.isna().mean() <= 0.5].copy()

    tmp_data = tmp_data.rename(
        columns={x: f"fea_{x}" for x in tmp_data.columns if x not in [item,'Accessions','region']}
    )

    feature_cols = [c for c in tmp_data.columns if c.startswith("fea_")]

    # =========================
    # Step1 先划分 train/test
    # =========================

    test_df = (
        tmp_data.groupby(["region"], group_keys=False)
        .apply(lambda x: x.sample(frac=0.1, random_state=42))
    )

    train_df = tmp_data.drop(test_df.index).copy()

    # =========================
    # Step2 用 train 计算 median
    # =========================

    train_group_median = (
        train_df.groupby(["region"])[feature_cols]
        .median()
    )

    global_median = train_df[feature_cols].median()

    # =========================
    # Step3 定义 fill 函数
    # =========================

    def fill_by_group(df, group_median, global_median):

        df = df.copy()

        for loc, idx in df.groupby(["region"]).groups.items():

            if loc in group_median.index:

                med = group_median.loc[loc]

                df.loc[idx, feature_cols] = df.loc[idx, feature_cols].fillna(med)

        df[feature_cols] = df[feature_cols].fillna(global_median)

        return df

    # =========================
    # Step4 填补缺失
    # =========================

    train_df = fill_by_group(train_df, train_group_median, global_median)
    test_df  = fill_by_group(test_df,  train_group_median, global_median)

    # =========================
    # Step5 标准化（只用train）
    # =========================

    scaler = preprocessing.StandardScaler()

    scaler.fit(train_df[feature_cols])

    train_df[feature_cols] = scaler.transform(train_df[feature_cols])
    test_df[feature_cols]  = scaler.transform(test_df[feature_cols])

    # =========================
    # 保存数据
    # =========================

    train_df.to_csv(
        os.path.join(WORK_PATH,f"trainTestData/trainFill-{item}.txt"),
        sep="\t",
        index=False
    )

    test_df.to_csv(
        os.path.join(WORK_PATH,f"trainTestData/testFill-{item}.txt"),
        sep="\t",
        index=False
    )

print("ALL DONE")

/tmp/ipykernel_276644/695581352.py:21: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda x: x.sample(frac=0.1, random_state=42))
/tmp/ipykernel_276644/695581352.py:21: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda x: x.sample(frac=0.1, random_state=42))
/tmp/ipykernel_276644/695581352.py:21: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is

ALL DONE


/tmp/ipykernel_276644/695581352.py:21: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda x: x.sample(frac=0.1, random_state=42))
